# Project 4: Release evidence

## 1. What can happen?

A report may look good but describe a different release or cover too few examples.

**Example** The current assistant is hr-v1, but a passing report belongs to hr-v0.

## 2. Our idea

We load four explicitly synthetic report cases and an example release policy. A release gate checks evidence before handing the release to a person for review. This lesson does not run an evaluation or deploy an application.

### How we do it

1. Inspect missing, failing, wrong-version, and passing reports.
2. Check version, dataset identity, coverage, accuracy, and observed privacy failures.
3. Replace only the wrong-version evidence selection with the existing current-version report and compare.

## 3. How it helps

A current passing report makes a release ready for human review. It provides no human approval. Zero privacy failures describes only this fixture's test set, not a guarantee.

## 4. Input and setup

Run uv sync --locked at the outer root. Open this numbered folder in VS Code,
select the root .venv/Scripts/python.exe kernel, and run one cell at a time.
All source examples are fictional and make no model calls.


### Step 1: Check the lesson folder

This cell only checks the working directory and imports standard-library tools.

In [ ]:
import json
from copy import deepcopy
from datetime import date
from pathlib import Path

project_dir = Path.cwd()
assert project_dir.name == "4_release_evidence", "Open this numbered folder in VS Code."
as_of = date(2026, 11, 15)
print("Lesson:", project_dir.name, "| fixed sample date:", as_of)

### Step 2: Read the report fixtures

The counts are invented teaching inputs, not measured model performance. Inspect both denominators and the required dataset identities.

In [ ]:
release = json.loads((project_dir / "docs/dataset/release.json").read_text())
policy = json.loads((project_dir / "docs/dataset/policy.json").read_text())
reports = json.loads((project_dir / "docs/dataset/reports.json").read_text())
print(json.dumps({"policy": policy, "reports": reports}, indent=2))

### Step 3: Apply the example release gate

Read tools/evidence.py. check_evidence checks report presence and valid counts before comparing identity, coverage, and thresholds. check_cases repeats this for the four samples.

In [ ]:
from tools.evidence import check_cases, show_results

baseline = check_cases(release, reports, policy)
show_results(baseline)

### Step 4: Select matching evidence

Replace one case's selected report with the existing current-version report. We do not change its release ID, fabricate new results, or lower a threshold.

In [ ]:
candidate_reports = deepcopy(reports)
candidate_reports["wrong_version"] = deepcopy(reports["passing"])
candidate = check_cases(release, candidate_reports, policy)
show_results(candidate)

### Step 5: Compare the same cases

Find the single changed decision. Missing evidence and genuine failures still block review.

In [ ]:
for name in baseline:
    print(name, baseline[name]["decision"], "->", candidate[name]["decision"])

### Step 6: Interpret the passing report

94/100 is the fixture's quality score. 0/20 is its observed privacy failure count. Neither result is a real evaluation performed by this notebook, and human approval remains pending.

In [ ]:
report = reports["passing"]
print("Synthetic quality counts:", report["correct"], "/", report["total"])
print("Synthetic observed privacy failures:", report["privacy_failures"], "/", report["privacy_tests"])
print("Human approval: pending. Deployment performed: no.")

### Explain the result

Initially only passing is ready_for_human_review. Selecting the current-version report for wrong_version makes that case ready too. Missing and failing cases stay blocked. Do not relabel an old report as fresh evidence.

A current passing report makes a release ready for human review. It provides no human approval. Zero privacy failures describes only this fixture's test set, not a guarantee.

Changes are in memory. This notebook writes no files.

Pause here. Your run and any human decisions remain your learning steps.